# Transcribe many recordings and calculate the error rate

In [notebook 1](1_first_api_call.ipynb) you transcribed one recording at a time.
Here you transcribe many recordings through the same API, save the transcripts,
and calculate how many words the model got wrong.

**One recording is one API call.** The Finnish test set has 916 recordings, so
transcribing all of it means 916 calls, sent one after another. That takes a
while, so start with a few recordings.

**Results are saved as you go.** Each transcript is written to a file as soon as
it arrives. If the notebook stops, or some calls fail, run section 5 again: it
only sends the recordings that are still missing.

## 1. Imports

In [1]:
import glob
import json
import os
import re
import time
import unicodedata
from getpass import getpass, getuser

import openai
import pandas as pd
from datasets import load_from_disk
from IPython.display import Audio

## 2. Settings

Start with `MAX_ENTRIES = 10` to check that everything works. Then set it to
`None` and run the notebook again: recordings already done are not sent again.

In [29]:
AITTA_BASE_URL = "https://aitta-api.csc.fi/openai/v1"   # Aitta's OpenAI-compatible API
ASR_MODEL = "openai/whisper-large-v3"                     # the speech-to-text model on Aitta

DATASET = "fleurs-fi"   # data name from run-prepare-data-lumi.sh ("fleurs-en" for English)
SPLIT = "test"          # "train", "validation" or "test"

# Where the prepared data is. None = find it on LUMI automatically.
# On your own computer, give the folder that contains data/<DATASET>, e.g. "~/asr"
WORK_DIR = None

MAX_ENTRIES = None       # how many recordings to transcribe; None = all of them

## 3. Open the prepared data

The results go into their own folder next to the data, named after the model,
dataset and split.

In [30]:
# Find the folder where the course scripts saved the data:
# /scratch/<project>/<username>/asr on LUMI, or WORK_DIR if you set it above.
if WORK_DIR is None:
    found = glob.glob(f"/scratch/project_*/{getuser()}/asr")
    if not found:
        raise FileNotFoundError("No prepared data found. Prepare it first with "
                                "run-prepare-data-lumi.sh, or set WORK_DIR above.")
    WORK_DIR = found[0]
WORK_DIR = os.path.expanduser(WORK_DIR)

# Opening the dataset is instant: each entry is read from disk only when used
ds = load_from_disk(os.path.join(WORK_DIR, "data", DATASET, SPLIT))
print(f"{len(ds)} recordings in {DATASET} ({SPLIT})")

OUT_DIR = os.path.join(WORK_DIR, "runs", "api", f"aitta-{DATASET}-{SPLIT}")
RESULTS = os.path.join(OUT_DIR, "transcriptions.jsonl")   # one line per recording
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Results go to {OUT_DIR}")

916 recordings in fleurs-fi (test)
Results go to /scratch/project_462000131/hintsala/asr/runs/api/aitta-fleurs-fi-test


## 4. Connect to Aitta

The access token is read from the environment variable `AITTA_API_KEY` if it is
set; otherwise you are asked to type it. It is not shown or saved in the notebook.

`max_retries=5`: if a call fails temporarily, the client waits and tries again by itself.

In [4]:
token = os.environ.get("AITTA_API_KEY") or getpass("Aitta access token: ")
client = openai.OpenAI(api_key=token, base_url=AITTA_BASE_URL, max_retries=5)

Aitta access token:  ········


## 5. Transcribe

The loop sends the recordings one by one. Each result is written to
`transcriptions.jsonl` right away, so nothing is lost if the notebook stops.
Recordings that are already in the file are skipped.

If some calls fail, they are listed at the end: **run this cell again** to retry them.

In [ ]:
# Which recordings are already done (from an earlier run)?
done = set()
if os.path.exists(RESULTS):
    with open(RESULTS, encoding="utf-8") as f:
        done = {json.loads(line)["id"] for line in f}

ids = ds["id"]
n = len(ds) if MAX_ENTRIES is None else min(MAX_ENTRIES, len(ds))
todo = [i for i in range(n) if ids[i] not in done]
print(f"{n} recordings selected, {n - len(todo)} already done, {len(todo)} API calls to make")

failed = []
start = time.time()
with open(RESULTS, "a", encoding="utf-8") as out:
    for k, i in enumerate(todo, 1):
        example = ds[i]
        try:
            # One API call: send the WAV file to the model
            transcription = client.audio.transcriptions.create(
                file=("audio.wav", example["audio"], "audio/wav"),
                model=ASR_MODEL,
                response_format="verbose_json",
            )
        except Exception as e:
            failed.append(example["id"])
            print(f"  FAILED {example['id']}: {e}")
            continue

        # Save the result right away, one line per recording
        result = {"id": example["id"],
                  "reference": example["text"],
                  "transcript": transcription.text.strip(),
                  "language": transcription.language}
        out.write(json.dumps(result, ensure_ascii=False) + "\n")
        out.flush()

        if k % 10 == 0 or k == len(todo):
            print(f"  {k}/{len(todo)} done ({time.time() - start:.0f} s)")

print(f"{len(todo) - len(failed)} transcribed, {len(failed)} failed. Saved in {RESULTS}")

916 recordings selected, 30 already done, 886 API calls to make
  10/886 done (20 s)
  20/886 done (38 s)
  30/886 done (59 s)
  40/886 done (78 s)
  50/886 done (108 s)
  60/886 done (129 s)
  70/886 done (148 s)
  80/886 done (166 s)
  90/886 done (187 s)
  100/886 done (207 s)
  110/886 done (226 s)
  120/886 done (246 s)
  130/886 done (267 s)
  140/886 done (288 s)
  150/886 done (305 s)
  160/886 done (323 s)
  170/886 done (344 s)
  180/886 done (364 s)
  190/886 done (384 s)
  200/886 done (405 s)
  210/886 done (427 s)
  220/886 done (447 s)
  230/886 done (464 s)
  240/886 done (489 s)
  250/886 done (506 s)
  260/886 done (527 s)
  270/886 done (544 s)
  280/886 done (567 s)
  290/886 done (586 s)
  300/886 done (605 s)


## 6. How the error rate is calculated

The **word error rate (WER)** compares the model's transcript with the correct
text in three steps:

1. **Normalize both texts:** lowercase, remove punctuation, collapse spaces. Only
   the words count, not how the model formats them.
2. **Count the edits:** the smallest number of word replacements, deletions and
   insertions that turn the correct text into the transcript.
3. **Divide** the edits by the number of words in the correct text.

The **character error rate (CER)** is the same with characters instead of words.
It matters for Finnish: one wrong letter in a long compound word makes the whole
word wrong in WER, but is only one wrong character in CER.

This is the same calculation as in the course scripts (`asr_common.py`), so the
results can be compared directly.

In [ ]:
def normalize_text(text):
    """Lowercase, remove punctuation (keep letters, digits and apostrophes inside
    words), and collapse spaces."""
    text = unicodedata.normalize("NFKC", text).lower().replace("_", " ")
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"(^|\s)'+|'+(\s|$)", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def edit_distance(ref, hyp):
    """The smallest number of replacements, deletions and insertions that turn
    the list ref into the list hyp (of words or of characters)."""
    prev = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        cur = [i] + [0] * len(hyp)
        for j, h in enumerate(hyp, 1):
            cur[j] = min(prev[j] + 1,              # deletion
                         cur[j - 1] + 1,           # insertion
                         prev[j - 1] + (r != h))   # replacement (or match)
        prev = cur
    return prev[-1]


def wer_cer(references, hypotheses):
    """WER and CER over all sentences together."""
    word_errors = words = char_errors = chars = 0
    for ref, hyp in zip(references, hypotheses):
        ref, hyp = normalize_text(ref), normalize_text(hyp)
        word_errors += edit_distance(ref.split(), hyp.split())
        words += len(ref.split())
        char_errors += edit_distance(list(ref), list(hyp))
        chars += len(ref)
    return {"wer": word_errors / max(words, 1), "cer": char_errors / max(chars, 1)}

Try it on an example, and change the sentences to see how the numbers react:

In [ ]:
correct = "Miljoonia gallonia polttoainetta säästyy vuosittain."
model   = "miljoonia kalonia polttoainetta säästyy vuosittain"

print("Normalized correct :", normalize_text(correct))
print("Normalized model   :", normalize_text(model))
scores = wer_cer([correct], [model])
print(f"WER {scores['wer']:.0%}   CER {scores['cer']:.0%}")

## 7. Score the transcripts and save them

The results are read into a table, scored, and saved as `side_by_side.tsv`, which
opens in Excel or LibreOffice: one row per sentence with the correct text, the
transcript and that sentence's WER.

In [ ]:
# Read the saved results into a table, in dataset order
df = pd.read_json(RESULTS, lines=True)
df = df[df["id"].isin(ids[:n])].sort_values("id").reset_index(drop=True)

# WER for each sentence, and for all sentences together
df["wer"] = [wer_cer([r], [h])["wer"] for r, h in zip(df["reference"], df["transcript"])]
total = wer_cer(df["reference"], df["transcript"])
print(f"{len(df)} sentences:  WER {total['wer']:.2%}   CER {total['cer']:.2%}")

# Save the table (utf-8-sig so that Excel shows ä and ö correctly)
df.to_csv(os.path.join(OUT_DIR, "side_by_side.tsv"), sep="\t", index=False, encoding="utf-8-sig")
print(f"Saved {os.path.join(OUT_DIR, 'side_by_side.tsv')}")

## 8. Look at the results

Sorting by WER shows the sentences the model got most wrong. A sentence's WER can
be **over 100%**: added words count as errors too, so a transcript can have more
errors than the correct text has words.

In [ ]:
pd.set_option("display.max_colwidth", None)     # show whole sentences
df.sort_values("wer", ascending=False)[["id", "wer", "reference", "transcript"]].head(10)

How many sentences were perfect, and which language did the model detect? The
model wasn't told the language, so a wrong guess shows up here.

In [26]:
print(f"Sentences without errors: {(df['wer'] == 0).mean():.0%}")
df["language"].value_counts()

Sentences without errors: 23%


language
fi    30
Name: count, dtype: int64

## 9. Listen to the sentence with the most errors

Hearing the recording often explains the mistake: unclear speech, a foreign name,
background noise, or even an error in the correct text.

In [27]:
worst = df.sort_values("wer", ascending=False).iloc[0]
print(f"{worst['id']}  (WER {worst['wer']:.0%})")
print(f"Correct : {worst['reference']}")
print(f"Model   : {worst['transcript']}")

Audio(ds[ids.index(worst["id"])]["audio"])

test-000024-1858  (WER 40%)
Correct : YK:n tautiasiantuntija Danielle Lantagne totesi rauhanturvaajien todennäköisesti aiheuttaneen epidemian.
Model   : YK on tautiasiantuntija Daniele Lantamäe totesi rauhanturvaihien todennäköisesti aiheuttaneen epidemian.


## Summary

- One recording is one API call; the results are saved as they arrive, so a long
  run can be stopped and continued.
- WER counts the word errors after normalizing both texts; CER does the same for
  characters.
- `side_by_side.tsv` shows *which* sentences go wrong, which tells more than the
  overall WER alone. Like the course scripts' comparison files, it has one row per
  sentence, so you can open them next to each other.

**Note:** the audio is sent to an external service. That is fine for public data
like FLEURS, but check Aitta's terms before sending your own or sensitive
recordings.